# CUDA × LLM 推理教程 — Colab 一键启动

运行步骤：
1. Runtime → Change runtime type → **T4 GPU**（或更高）
2. 从上到下依次运行所有 cell
3. 训练/推理示例会自动落在 `/content/ops` 目录

**预计时间**：环境准备 ~2 分钟；编译全部 14 章 ~5 分钟。

In [1]:
# 1) 检查 GPU
!nvidia-smi | head -20
!nvcc --version

Wed Jul  8 07:29:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2) 克隆仓库
import os
if not os.path.exists('/content/ops'):
    # TODO: replace with your fork URL after pushing
    !git clone https://github.com/hswsp/learn-cuda-from-scratch.git /content/ops
%cd /content/ops

Cloning into '/content/ops'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: '/content/ops'
/content


In [ ]:
# 3) 选择 GPU 架构
# Colab Free T4 → sm_75, A100 → sm_80, L4 → sm_89
import subprocess
out = subprocess.check_output('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True).decode().strip()
print('detected GPU:', out)
ARCH = 'sm_75' if 'T4' in out else ('sm_89' if 'L4' in out else 'sm_80')
print('using ARCH =', ARCH)

In [ ]:
# 4) 编译并跑通第 2 章 Hello CUDA
!make -C code/ch02_hello ARCH={ARCH} clean all run

In [ ]:
# 5) 跑核心章节示例（Ch6 Tile / Ch9 GEMM / Ch12 FlashAttention）
for ch in ['ch06_tile', 'ch09_gemm', 'ch12_flashattn']:
    print(f'==> {ch}')
    !make -C code/{ch} ARCH={ARCH} run | tail -30

In [ ]:
# 6) 下载 GPT-2 small 权重（Capstone 用，~250MB）
!pip install -q transformers torch
!python data/download_gpt2.py --out data/gpt2-small.bin

In [ ]:
# 7) 跑 Capstone：用自己写的 kernel 做 GPT-2 推理
!make -C code/ch14_mini_llm ARCH={ARCH} clean all
!./code/ch14_mini_llm/mini_llm --weights data/gpt2-small.bin --prompt 'Once upon a time'